In [32]:
import numpy as np
import torch

def minmax_normalization(values, max_values, min_values):
    if min_values != max_values:
        val = (values - min_values) / (max_values - min_values)
    else:
        val = 0  # Skip the Feature
    return val
    
def psnr_error(gen_frames, gt_frames, hat=False):
    """
    Computes the Peak Signal to Noise Ratio error between the generated images and the ground
    truth images.
    @param gen_frames: A tensor of shape [batch_size, height, width, 3]. The frames generated by the
                       generator model.
    @param gt_frames: A tensor of shape [batch_size, height, width, 3]. The ground-truth frames for
                      each frame in gen_frames.
    @return: A scalar tensor. The mean Peak Signal to Noise Ratio error over each frame in the batch.
    """
    gen_frames = gen_frames.detach().cpu()
    gt_frames = gt_frames.detach().cpu()
    batch_num = gen_frames.shape[0]
    batch_errors = 0.0
    for i in range(batch_num):
        num_pixels = gen_frames[i].numel()
        # max_val_hat = gen_frames[i].max()
        if hat:
            max_val = gen_frames[i].max()
        else:
            max_val = gt_frames[i].max()
        square_diff = (gt_frames[i] - gen_frames[i]) ** 2
        log_value = torch.log10(max_val / ((1. / num_pixels) * torch.sum(square_diff)))
        image_errors = 10 * log_value
        batch_errors += image_errors

    batch_errors = torch.div(batch_errors, batch_num)
    return batch_errors


def simple_diff(frame_true, frame_hat, aggregation=False):
    """
    """
    assert frame_true.shape == frame_hat.shape
    frame_true = frame_true.squeeze(0).detach()
    frame_hat = frame_hat.squeeze(0).detach()
    loss_appe = (frame_true - frame_hat) ** 2
    if aggregation:
        loss_appe = torch.mean(loss_appe)
    return loss_appe


def find_max_patch(diff_map_appe, kernel_size=16, stride=4, aggregation=True):
    """
    kernel size = window size
    """
    # max_pool = torch.nn.MaxPool2d(kernel_size=kernel_size, stride=stride)
    avg_pool = torch.nn.AvgPool2d(kernel_size=kernel_size, stride=stride)
    max_patch_appe = avg_pool(diff_map_appe)
    print('max_patch_appe.shape =', max_patch_appe.shape)
    # import ipdb; ipdb.set_trace()
    assert len(max_patch_appe.shape) == 3, f'the shape of max_patch_appe is {max_patch_appe.shape}'

    if aggregation:
        # Will sum the channel dim
        max_patch_appe = torch.mean(max_patch_appe, dim=0)

    max_appe_value = torch.max(max_patch_appe)

    app_h, app_w = torch.where(torch.eq(max_patch_appe, max_appe_value))

    max_appe_final = max_appe_value
    return max_appe_final, (app_h, app_w)


def calc_w(w_dict):
    wi = 0.0
    n = 0
    for key in w_dict.keys():
        # n += w_dict[key][0]
        n += 1
        wi += w_dict[key][1]
    # import ipdb; ipdb.set_trace()
    wi = torch.div(1.0, torch.div(wi, n))

    return wi


def amc_normal_score(wi, si):
    final_score = torch.log(wi * si)
    #final_score = torch.log10(wi * si)
    #print('wi = ', wi)
    print('si = ', si)
    return final_score


def amc_score(frame, frame_hat, wi, kernel_size=16, stride=4):
    """
    wf, wi is different from videos
    """
    loss_appe = simple_diff(frame, frame_hat)
    max_patch_appe, app_cord = find_max_patch(loss_appe, kernel_size=kernel_size,
                                              stride=stride)
    final_score = amc_normal_score(wi, max_patch_appe)

    return final_score, app_cord


In [21]:
#from utils_AMC import *
from collections import OrderedDict
import torch

num_videos = 2
num_clips = 3

train_loader = []
test_loader = []
# Repeat the initialization process 5 times
for idx in range(num_videos):
    train_target = []  # Reset train_target for each video
    train_output = []  # Reset train_output for each video
    test_target = []
    test_output = []
    for i in range(num_clips):
        # Create a tensor with the specified size
        train_target.append(torch.randn(1, 3, 256, 256))
        train_output.append(torch.randn(1, 3, 256, 256)) 
        
        test_target.append(torch.randn(1, 3, 256, 256))
        test_output.append(torch.randn(1, 3, 256, 256))
    train_loader.append((train_target, train_output))
    test_loader.append((test_target, test_output))

# train_loader
w_dict = OrderedDict()
len_dataset = len(train_loader)
idx = 0
for clips_of_video in train_loader:
    train_target, train_output = clips_of_video
    patch_scores = []
    for target, output in zip(train_target, train_output):
        # Access individual tensors and print their shapes
        # print('train_target.shape = ', target[0].shape)
        # print('train_output.shape = ', output[0].shape)

        diff_appe = simple_diff(target, output)
        # print('diff_appe', diff_appe)
        patch_score_appe, (app_h, app_w) = find_max_patch(diff_appe)
        patch_scores.append(patch_score_appe)
        print('patch_score_appe', patch_score_appe)
        print(f'app_h ={app_h}, app_w ={app_w}')

    patch_scores = torch.tensor(patch_scores)
    patch_scores = torch.mean(patch_scores)
    print('patch_scores', patch_scores)
    
    #frame_w = torch.mean(patch_scores)
    #print('frame_w', frame_w)

    w_dict[idx] = [len_dataset, patch_scores]
    idx += 1

wi = calc_w(w_dict)
print(f'wi:{wi}')

patch_score_appe tensor(2.4928)
app_h =tensor([47]), app_w =tensor([18])
patch_score_appe tensor(2.3824)
app_h =tensor([31]), app_w =tensor([54])
patch_score_appe tensor(2.3637)
app_h =tensor([59]), app_w =tensor([52])
patch_scores tensor(2.4129)
patch_score_appe tensor(2.4033)
app_h =tensor([31]), app_w =tensor([56])
patch_score_appe tensor(2.3669)
app_h =tensor([5]), app_w =tensor([55])
patch_score_appe tensor(2.3823)
app_h =tensor([38]), app_w =tensor([55])
patch_scores tensor(2.3842)
wi:0.41691938042640686


In [3]:
print('len(test_loader) =', len(test_loader))

len(test_loader) = 2


In [36]:
import numpy as np

video_psnr = []
video_score = []
wi =1
for clips_of_video in test_loader:
    test_target, test_output = clips_of_video
    frame_psnr = []
    frame_pc = []
    for target, output in zip(test_target, test_output):
        # Access individual tensors and print their shapes
        # print('train_target.shape = ', target[0].shape)
        # print('train_output.shape = ', output[0].shape)

        psnr = psnr_error(output, target)
        psnr = psnr.tolist()
        frame_psnr.append(psnr)
        #print('psnr = ', psnr)
        
        pc, app_cord = amc_score(target, output, wi)
        pc = pc.tolist()
        frame_pc.append(pc)
        #print('pc = ', pc)

    print("============= pc_scores =============")
    print('frame_pc = ', frame_pc)
    smax = max(frame_pc)
    print('smax = ', smax)
    
    pc_scores = np.array([np.divide(s, smax) for s in frame_pc])
    print('pc_scores = ', pc_scores)

    # pc_scores must be >= 0
    pc_scores = np.clip(pc_scores, 0, None)
    print('pc_scores is clipped = ', pc_scores)

    print("===>>> psnr_scores")
    print('frame_psnr = ', frame_psnr)
    # frame_psnr
    psnr_scores = minmax_normalization(frame_psnr,
                        np.max(frame_psnr),
                        np.min(frame_psnr))
    print('1 - psnr_scores = ', 1- psnr_scores)
    """
    for i in range(len(frame_psnr)):
        print(f'frame_psnr[{i}] = {"{:.2f}".format(frame_psnr[i])}') 
    for i in range(len(psnr_scores)):
        print(f'psnr_scores[{i}] = {"{:.2f}".format(psnr_scores[i])}')  
    """
    video_psnr.append(psnr_scores)
    video_score.append(pc_scores)

max_patch_appe.shape = torch.Size([3, 61, 61])
si =  tensor(2.3515)
max_patch_appe.shape = torch.Size([3, 61, 61])
si =  tensor(2.3853)
max_patch_appe.shape = torch.Size([3, 61, 61])
si =  tensor(2.3499)
============= pc_scores =============
frame_pc =  [0.8550468683242798, 0.869329571723938, 0.8543685078620911]
smax =  0.869329571723938
pc_scores =  [0.98357044 1.         0.98279011]
pc_scores is clipped =  [0.98357044 1.         0.98279011]
===>>> psnr_scores
frame_psnr =  [3.267287015914917, 2.9786486625671387, 3.6088767051696777]
1 - psnr_scores =  [0.54200966 1.         0.        ]
max_patch_appe.shape = torch.Size([3, 61, 61])
si =  tensor(2.3608)
max_patch_appe.shape = torch.Size([3, 61, 61])
si =  tensor(2.3731)
max_patch_appe.shape = torch.Size([3, 61, 61])
si =  tensor(2.3165)
============= pc_scores =============
frame_pc =  [0.859010636806488, 0.8642174005508423, 0.8400670886039734]
smax =  0.8642174005508423
pc_scores =  [0.99397517 1.         0.97205528]
pc_scores is clip